<h2 style="color:#ff7800;">Complete Exercise</h2>
            <span style="color:#ff7800;">1. Add multiple LLM calls! After the LLM forms its reply, use another LLM call to evaluate that it is strictly related to work only.<br/><br/>2. Apply this to your business! Make an AI Assistant that can answer questions about your business area, and use the tool to record email addresses of people who want to get in touch.
            </span>

In [ ]:
from dotenv import load_dotenv
from openai import OpenAI
from pypdf import PdfReader
from IPython.display import Markdown, display
import gradio as gr
import json

In [ ]:
load_dotenv(override=True)
openai = OpenAI()

In [ ]:
reader = PdfReader("twin/linkedin.pdf")
linkedin = ""
for page in reader.pages:
    text = page.extract_text()
    if text:
        linkedin += text

In [ ]:
print(linkedin)

In [ ]:
with open("twin/summary.txt", "r", encoding="utf-8") as f:
    summary = f.read()

In [ ]:
print(summary)

## Sidebar: Three concepts as a refresher

1. System Prompt: the part of the input to the LLM that describes the overall context of the conversation

2. Conversation History: the complete conversation so far

3. The illusion of memory: every message to an LLM is stateless. We pass in the complete conversation so far to give the illusion that it remembers what was said 30 seconds ago...


In [ ]:
messages = [
    {"role": "system", "content": "You are a helpful assistant"},
    {"role": "user", "content": "Hi, my name is Karina"}
]

In [ ]:
response = openai.chat.completions.create(model="gpt-5.4-nano", messages=messages)
print(response.choices[0].message.content)

## Main plot!

We have a LinkedIn profile in variable `linkedin`

We have a summary in variable `summary`

Let's construct a System Prompt..

In [ ]:
system_prompt = f"""

# Your role

You are a digital twin running on a website and chatting with visitors.
You represent the person whose website you are on.

You answer questions about this person's career, professional background,
skills, experience, and work.

Here are the details of the person you are representing:

{summary}

If asked, clearly explain that you are an AI digital twin of this person.

# Context

Here is a summary of the person's LinkedIn profile that you can use to answer questions:

{linkedin}

# Rules

Engage with the user in a professional, natural, and friendly way, as if you were
speaking with a potential client, recruiter, collaborator, or future employer
who came across the website.

Keep the conversation focused on Karina's professional background, career,
skills, experience, and work.

Greetings, farewells, and brief polite conversation are allowed.

Do not reveal or discuss personal information about Karina unless it is directly
relevant to her professional background or work.

If the user asks about something unrelated to Karina's professional background,
politely steer the conversation back to professional topics.

Always stay in character as Karina's digital twin.

IMPORTANT: If you do not know the answer, say so.
Never invent information.
If the user asks about something that is not supported by the provided context,
say that you do not know.

If the user asks how to contact Karina, expresses interest in a professional
opportunity, or would like Karina to contact them, provide the available
professional contact information and offer to record their email address so
Karina can get in touch with them.

Only call the email-recording tool after the user explicitly provides an email
address.
Do not invent, infer, or modify an email address.
"""

In [ ]:
messages = [
    {"role": "system", "content": system_prompt},
    {"role": "user", "content": "Hi - please tell me about yourself"},
]

In [ ]:
response = openai.chat.completions.create(model="gpt-5.4-mini", messages=messages)
display(Markdown(response.choices[0].message.content))

In [ ]:
def chat(message, history):
    messages = [{"role": "system", "content": system_prompt}] + history + [{"role": "user", "content": message}]
    response = openai.chat.completions.create(model="gpt-5.4-mini", messages=messages)
    return response.choices[0].message.content

In [ ]:
gr.ChatInterface(chat).launch(inbrowser=True)

# And now - TOOLS!

Let's start with a function... The original function doesn't check if the email already exists,here I added a little improve to avoid that,

In [ ]:
def record_email_tool(email):
    print(f"Tool called to record an email: {email}")

    try:
        with open("emails.txt", "r", encoding="utf-8") as f:
            emails = {line.strip().lower() for line in f if line.strip()}
    except FileNotFoundError:
        emails = set()

    normalized_email = email.strip().lower()

    if normalized_email in emails:
        return "Email already exists"

    with open("emails.txt", "a", encoding="utf-8") as f:
        f.write(normalized_email + "\n")

    return "Email received"

In [ ]:
record_email_tool("test@testy.com")

# Write some json to describe the tool

In [ ]:
record_email_tool_json = {
    "name": "record_email_tool",
    "description": "Use this tool to record that a user provided their email address",
    "parameters": {
        "type": "object",
        "properties": {
            "email": {"type": "string", "description": "The email address of this user"}
        },
        "required": ["email"],
        "additionalProperties": False
    }
}

In [ ]:
tools = [{"type": "function", "function": record_email_tool_json}]

# A new chat() function
To resolve the exercise I added a un evaluator to check if the response is enterely professional

In [ ]:
def evaluate_response(user_message, assistant_response):
  evaluator_prompt = evaluator_prompt = f"""
    You are an evaluator for a work-focused assistant.

    User message:
    {user_message}

    Assistant response:
    {assistant_response}

    The assistant should remain focused on work-related topics.

    Greetings, farewells, and brief polite conversation are allowed
    and should not be considered off-topic.

    The assistant should be cordial and natural, but it must not reveal
    or discuss personal information about Karina unless that information
    is directly relevant to her professional background, skills,
    experience, or work.

    If the user asks for personal information that is not work-related,
    the assistant should politely refuse and redirect the conversation
    toward professional topics.

    If the user message is work-related, evaluate whether the assistant's
    response is relevant, professional, and accurate.

    Return JSON only:
    {{
        "approved": true,
        "feedback": ""
    }}

    or:

    {{
        "approved": false,
        "feedback": "Specific instructions for improving the response"
    }}
    """
  
  messages = [{"role": "system", "content": evaluator_prompt}]
  response = openai.chat.completions.create(model="gpt-5.4-nano", messages=messages)
  return json.loads(response.choices[0].message.content)

In [ ]:
def chat(message, history):
    messages = [{"role": "system", "content": system_prompt}] + history + [{"role": "user", "content": message}]
    response = openai.chat.completions.create(model="gpt-5.4-mini", messages=messages, tools=tools)

    while response.choices[0].finish_reason=="tool_calls":
      message = response.choices[0].message
      messages.append(message)
      for tool_call in message.tool_calls:
          email = json.loads(tool_call.function.arguments).get("email")
          record_email_tool(email)
          messages.append({"role": "tool", "content": "Email recorded", "tool_call_id": tool_call.id})
      response = openai.chat.completions.create(model="gpt-5.4-mini", messages=messages, tools=tools)
          
    final_response = response.choices[0].message.content

    evaluation = evaluate_response(message, final_response)
    if not evaluation["approved"]:
        feedback = evaluation["feedback"]

        messages.append({
            "role": "assistant",
            "content": final_response
        })

        messages.append({
            "role": "system",
            "content": (
                "Improve your previous response based on this evaluator "
                f"feedback: {feedback}. "
                "The final response must be strictly related to work."
            )
        })

        response = openai.chat.completions.create(
            model="gpt-5.4-mini",
            messages=messages,
            tools=tools
        )

        final_response = response.choices[0].message.content

    return final_response

In [ ]:
gr.ChatInterface(chat).launch(inbrowser=True)